# GYMTEC — Resultados del pipeline ML (branch `feature/ml`)

Este notebook recorre **todo lo que implementé** y te muestra los outputs.
Si solo querés ver los resultados, dale **Run All** y revisa cada sección.

## Mapa del pipeline

```
data/raw/*.xlsx  ─▶  src/data/clean_data.py        ─▶  data/interim/   (SILVER)
                 ─▶  src/features/build_features.py ─▶  data/processed/ (GOLD)
                 ─▶  src/models/train_model.py      ─▶  models_artifacts/
                 ─▶  src/models/predict_model.py    ─▶  predicciones_aforo.parquet
                 ─▶  src/recommendation/...         ─▶  recomendaciones_horario.parquet
```

Trazabilidad: RF-01, RF-02, RF-03, RF-04, RF-06, RF-07.

In [ ]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Ejecutar el pipeline end-to-end

Limpia datos → genera features → entrena modelo → predice → recomienda.
Se puede correr cuantas veces quieras, es idempotente (RF-07).

In [ ]:
from src.run_pipeline import main as run_pipeline
run_pipeline()

## 2. Capa SILVER — datos limpios

**`logs_limpios.parquet`**: logs del gym sin sesiones <16 min, con
ocupación acumulada por día.

**`horarios_limpios.parquet`**: horarios académicos parseados
(día, hora_inicio, hora_fin, facultad, modalidad, clase_id).

In [ ]:
from src.utils import paths

logs = pd.read_parquet(paths.LOGS_LIMPIOS_PARQUET)
horarios = pd.read_parquet(paths.HORARIOS_LIMPIOS_PARQUET)

print(f'logs_limpios:     {logs.shape[0]:>5} filas × {logs.shape[1]} columnas')
print(f'horarios_limpios: {horarios.shape[0]:>5} filas × {horarios.shape[1]} columnas')
print()
print('--- logs_limpios (head) ---')
print(logs.head())
print()
print('--- horarios_limpios (head) ---')
print(horarios.head())

## 3. Capa GOLD — feature store

**`aforo_por_slot.parquet`**: serie temporal de aforo real (la base para
el target del modelo).

**`features_aforo_rf01.parquet`**: tabla principal con 30 columnas
(target + lags + cíclicas + variables académicas).

In [ ]:
aforo = pd.read_parquet(paths.AFORO_POR_SLOT_PARQUET)
features = pd.read_parquet(paths.FEATURES_AFORO_PARQUET)

print(f'aforo_por_slot:       {aforo.shape}')
print(f'features_aforo_rf01:  {features.shape}')
print()
print('--- aforo_por_slot (head) ---')
print(aforo.head())
print()
print('--- features_aforo_rf01 (head, columnas clave) ---')
cols_clave = ['fecha','dia','slot','aforo','ratio_ocupacion','nivel_ocupacion',
              'aforo_lag1','aforo_prom_hist','carga_academica','ratio_libres',
              'modalidad_predominante','es_dia_academico']
print(features[cols_clave].head(10))

### 3.1 Origen de cada variable

| Bloque | Columnas | De dónde sale |
|---|---|---|
| Identificadores | `fecha, dia, dia_num, slot, hora_dec` | derivado |
| Target / niveles | `aforo, aforo_max, ratio_ocupacion, nivel_ocupacion` | log_gym |
| Lags | `aforo_lag1, aforo_lag2` | feature engineering |
| Históricos | `aforo_prom_hist, aforo_max_hist` | feature engineering |
| Cíclicas | `hora_sin, hora_cos, dia_sin, dia_cos, es_finde` | feature engineering |
| Académicas | `estudiantes_presencial, estudiantes_libres, ratio_libres, carga_academica, ratio_virtual` | horarios_clases |
| Cardinalidades | `n_secciones_activas, n_cursos_activos, n_facultades_activas, duracion_prom_clases` | horarios_clases |
| Modalidad | `modalidad_predominante` | horarios_clases |

## 4. Modelo baseline RF-01

RandomForestRegressor con split temporal 80/20.

In [ ]:
metrics = json.loads(paths.MODEL_METRICS_JSON.read_text())

print('=== Métricas en test ===')
print(f"  Modelo: {metrics['model_kind']}")
print(f"  Train: {metrics['n_train']} slots  |  Test: {metrics['n_test']} slots")
print(f"  Corte temporal: {metrics['fecha_corte']}")
print(f"  MAE  = {metrics['mae']:.2f} personas")
print(f"  RMSE = {metrics['rmse']:.2f} personas")
print(f"  R²   = {metrics['r2']:.3f}")

In [ ]:
# Top 10 features más importantes
imp = pd.Series(metrics['feature_importances']).sort_values(ascending=True).tail(10)
fig, ax = plt.subplots(figsize=(9, 5))
imp.plot.barh(ax=ax, color='steelblue')
ax.set_title('Top 10 features más importantes (RF-01)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 5. RF-01 + RF-03 — heatmaps de aforo

Comparación visual entre el aforo histórico real y el aforo predicho por
el modelo sobre la grilla operativa del gym.

In [ ]:
predicciones = pd.read_parquet(paths.PREDICCIONES_AFORO_PARQUET)

orden = ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado']
pivot_real = (
    aforo.pivot_table(index='dia', columns='slot', values='aforo', aggfunc='mean')
         .reindex([d for d in orden if d in aforo['dia'].unique()])
)
pivot_pred = (
    predicciones.pivot_table(index='dia', columns='slot', values='aforo_predicho', aggfunc='first')
                .reindex([d for d in orden if d in predicciones['dia'].unique()])
)

fig, axes = plt.subplots(2, 1, figsize=(15, 8))
sns.heatmap(pivot_real.fillna(0), cmap='YlOrRd', annot=True, fmt='.0f',
            ax=axes[0], cbar_kws={'label':'aforo'})
axes[0].set_title('Aforo histórico promedio (RF-03)')
axes[0].set_xlabel('')

sns.heatmap(pivot_pred.fillna(0), cmap='YlOrRd', annot=True, fmt='.0f',
            ax=axes[1], cbar_kws={'label':'aforo predicho'})
axes[1].set_title('Aforo predicho por el modelo (RF-01)')
axes[1].set_xlabel('Slot horario')
plt.tight_layout()
plt.show()

## 6. RF-06 — distribución de niveles de ocupación

Cada slot se clasifica como `bajo / medio / alto / critico` según el ratio
aforo_predicho / aforo_max (umbrales 0.50 / 0.75 / 0.90).

In [ ]:
vc = predicciones['nivel_ocupacion'].value_counts()
print('Slots por nivel de ocupación:')
print(vc)

fig, ax = plt.subplots(figsize=(6, 4))
colors = {'bajo':'#22c55e','medio':'#facc15','alto':'#ef4444','critico':'#7f1d1d'}
vc.plot.bar(ax=ax, color=[colors.get(k,'gray') for k in vc.index])
ax.set_title('Distribución de niveles de ocupación predicha')
ax.set_ylabel('Slots')
plt.tight_layout()
plt.show()

## 7. RF-04 — franjas con menor aforo por día

Ranking de los slots con menor ocupación predicha para un día específico.
Es lo que un estudiante consultaría: *"hoy martes, ¿cuándo conviene ir?"*

In [ ]:
from src.recommendation.recommend_schedule import franjas_menor_aforo

for dia in ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado']:
    print(f'\n=== Top 5 franjas con menor aforo el {dia} ===')
    print(franjas_menor_aforo(dia, top_n=5)[['slot','aforo_predicho','nivel_ocupacion']])

## 8. RF-02 — recomendación personalizada (top 3)

Demostración con un estudiante demo. La función `recomendar_para_estudiante`
recibe el `student_id` y la lista de `clase_id` (cursos en los que está
matriculado), filtra los slots libres y devuelve los 3 con menor aforo.

In [ ]:
from src.recommendation.recommend_schedule import recomendar_para_estudiante

horarios_slots = pd.read_parquet(paths.HORARIOS_SLOTS_PARQUET)

# Estudiante demo: tomamos 5 cursos al azar como su carga académica
cursos_demo = horarios_slots['clase_id'].drop_duplicates().sample(5, random_state=42).tolist()
print('Cursos del estudiante demo:', cursos_demo)
print()

recos = recomendar_para_estudiante(
    student_id='ESTUDIANTE_DEMO',
    cursos_estudiante=cursos_demo,
    top_n=3,
)
for _, r in recos.iterrows():
    print(f"  #{int(r['ranking_recomendacion'])} {r['dia']} {r['slot']} "
          f"→ aforo {int(r['aforo_predicho'])}/50 ({r['nivel_ocupacion']})")
    print(f"     {r['razon_recomendacion']}")
    print()

## 9. Cómo consume el backend estos datos

Las apps `apps/student-app` y `apps/admin-web` hoy usan mocks. Cuando se
implemente el backend (`app/api/`), debería leer estos archivos:

| Endpoint que el backend expondrá | Lee de | Frontend que lo consume |
|---|---|---|
| `GET /predictions/today` | `predicciones_aforo.csv` | `student-app` (HomeScreen, PredictionScreen) |
| `GET /occupancy/heatmap` | `aforo_por_slot.parquet` | `admin-web` (AdminDashboardScreen) |
| `POST /recommendations` | corre `recomendar_para_estudiante` | `student-app` (OptimizeScreen) |
| `GET /low-occupancy?dia=X` | corre `franjas_menor_aforo` | `student-app` (ResultsScreen) |

Eso es todo. Los datos ya están en `data/processed/` listos para consumirse.